In [1]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler().set_output(transform='pandas')

# Original inputs ./f_results/{stroke,wids,diabetes}_results.csv are not in the repo; the shipped
# single-attribute ('age') grid-search outputs have the same layout (model, BM, metrics..., fair_score).
stroke_df = pd.read_csv('../script/results/aggregated_stroke_age.csv')
wids_df = pd.read_csv('../script/results/aggregated_wids_age.csv')
diabetes_df = pd.read_csv('../script/results/aggregated_diabetes_age.csv')

# Some rows contain +/-inf (e.g. disparate_impact when a group has no positives); MinMaxScaler cannot take inf, so treat as missing
import numpy as np
for _d in (stroke_df, wids_df, diabetes_df):
    _d.replace([np.inf, -np.inf], np.nan, inplace=True)

In [2]:
min_max_scaler = MinMaxScaler().set_output(transform='pandas')

df_l = [stroke_df, wids_df, diabetes_df]
for df in df_l:
    print(scaler.fit_transform(df[df.columns[2:]]))
    df[df.columns[2:]] = scaler.fit_transform(df[df.columns[2:]]) 

     balanced_acc       acc  precision  recall        f1       mcc  \
0        0.909238  0.485714   0.693878    0.85  0.805779  0.826166   
1        0.761194  0.514286   0.645161    0.70  0.709685  0.782500   
2        0.822912  0.442857   0.643863    0.80  0.740319  1.000000   
3        0.854377  0.328571   0.619621    0.90  0.740319  0.717963   
4        0.754336  0.742857   0.785714    0.55  0.753986  0.871905   
..            ...       ...        ...     ...       ...       ...   
114      0.319080  1.000000   0.000000    0.00       NaN       NaN   
115      0.319080  1.000000   0.000000    0.00       NaN       NaN   
116      0.319080  1.000000   0.000000    0.00       NaN       NaN   
117      0.840662  0.785714   0.879121    0.60  0.851367  0.659417   
118      0.319080  1.000000   0.000000    0.00       NaN  0.184770   

     eq_opp_diff  avg_odd_diff       spd  disparate_impact  theil_idx  \
0       0.233766      0.150318  0.105489          0.093059   0.199697   
1       0.324

In [3]:
import numpy as np
from scipy import stats
from statsmodels.stats.anova import anova_lm
from statsmodels.formula.api import ols

# Helper function to calculate relative changes
def calc_relative_changes(df):
    results = []
    
    # Get unique models
    models = df['model'].unique()
    
    for model in models:
        # Get baseline metrics for this model
        baseline = df[(df['model'] == model) & (df['BM'] == 'baseline')].iloc[0]
        
        # Get all configurations for this model
        configs = df[df['model'] == model]
        
        for _, config in configs.iterrows():
            if config['BM'] != 'baseline':
                # Calculate relative changes
                acc_change = (config['acc'] - baseline['acc']) / baseline['acc']
                fair_change = config['fair_score'] - baseline['fair_score']
                
                results.append({
                    'model': model,
                    'BM': config['BM'],
                    'acc_change': acc_change,
                    'fair_change': fair_change,
                    'balanced_acc_change': (config['balanced_acc'] - baseline['balanced_acc']) / baseline['balanced_acc'],
                    'dataset': 'diabetes',
                    'effectiveness': acc_change - fair_change  # Calculate effectiveness here for ANOVA
                })
    
    return pd.DataFrame(results)

# Calculate changes for both datasets
diabetes_changes = calc_relative_changes(diabetes_df)
stroke_changes = calc_relative_changes(stroke_df)
wids_changes = calc_relative_changes(wids_df)
diabetes_changes['dataset'] = 'diabetes'
stroke_changes['dataset'] = 'stroke'
wids_changes['dataset'] = 'wids'

# Combine results
all_changes = pd.concat([diabetes_changes, stroke_changes, wids_changes])
np.isinf(all_changes.select_dtypes('number')).sum()  # count of inf values per column

acc_change             0
fair_change            0
balanced_acc_change    0
effectiveness          0
dtype: int64

In [4]:
# Perform Two-way ANOVA analyses
# 1. For effectiveness (combined metric)
effectiveness_model = ols('effectiveness ~ C(model) + C(BM) + C(model):C(BM)', data=all_changes).fit()
effectiveness_anova = anova_lm(effectiveness_model, typ=2)

# 2. For accuracy changes
accuracy_model = ols('acc_change ~ C(model) + C(BM) + C(model):C(BM)', data=all_changes).fit()
accuracy_anova = anova_lm(accuracy_model, typ=2)

# 3. For fairness changes
fairness_model = ols('fair_change ~ C(model) + C(BM) + C(model):C(BM)', data=all_changes).fit()
fairness_anova = anova_lm(fairness_model, typ=2)

# Calculate effect sizes (partial eta-squared) for each ANOVA
def calculate_partial_eta_squared(aov):
    aov['pes'] = aov['sum_sq'] / (aov['sum_sq'] + aov['sum_sq'].sum())
    return aov

effectiveness_anova = calculate_partial_eta_squared(effectiveness_anova)
accuracy_anova = calculate_partial_eta_squared(accuracy_anova)
fairness_anova = calculate_partial_eta_squared(fairness_anova)

# Aggregate results by BM strategy
strategy_performance = all_changes.groupby('BM').agg({
    'acc_change': ['mean', 'std'],
    'fair_change': ['mean', 'std'],
    'effectiveness': ['mean', 'std']
}).round(4)

# Find best performing strategies
best_strategies = all_changes[
    (all_changes['acc_change'] > -0.1) & 
    (all_changes['fair_change'] < 0)
].groupby('BM')['effectiveness'].mean().sort_values(ascending=False)

# Add interaction analysis
# Calculate mean effectiveness for each model-BM combination
interaction_analysis = all_changes.pivot_table(
    values='effectiveness',
    index='model',
    columns='BM',
    aggfunc='mean'
).round(4)

d_val = {
    'strategy_performance': strategy_performance,
    'best_strategies': best_strategies,
    'effectiveness_anova': effectiveness_anova,
    'accuracy_anova': accuracy_anova,
    'fairness_anova': fairness_anova,
    'interaction_analysis': interaction_analysis,
    'detailed_changes': all_changes
}

# To interpret ANOVA results:
# - Low p-values (<0.05) in the ANOVA tables indicate significant effects
# - Partial eta-squared (pes) values indicate effect sizes:
#   - Small effect: ~0.01
#   - Medium effect: ~0.06
#   - Large effect: ~0.14

/var/folders/yb/kqtv5rps7b528n9jv4zm9pvw0000gn/T/ipykernel_60548/405655638.py:3: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  effectiveness_model = ols('effectiveness ~ C(model) + C(BM) + C(model):C(BM)', data=all_changes).fit()


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 9, but rank is 6
  res = self.wald_test(


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 30, but rank is 2
  res = self.wald_test(


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 270, but rank is 103
  res = self.wald_test(


/var/folders/yb/kqtv5rps7b528n9jv4zm9pvw0000gn/T/ipykernel_60548/405655638.py:7: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  accuracy_model = ols('acc_change ~ C(model) + C(BM) + C(model):C(BM)', data=all_changes).fit()


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 9, but rank is 6
  res = self.wald_test(


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 30, but rank is 1
  res = self.wald_test(


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 270, but rank is 103
  res = self.wald_test(
/var/folders/yb/kqtv5rps7b528n9jv4zm9pvw0000gn/T/ipykernel_60548/405655638.py:11: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  fairness_model = ols('fair_change ~ C(model) + C(BM) + C(model):C(BM)', data=all_changes).fit()


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 9, but rank is 6
  res = self.wald_test(


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 30, but rank is 1
  res = self.wald_test(


/Users/pedropedro/Downloads/callmefair/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:1988: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 270, but rank is 103
  res = self.wald_test(


In [5]:
d_val['interaction_analysis']

BM,inAD,inAD posCalibrated,inAD posEqqOds,inAD posROC,posCalibrated,posEqqOds,posROC,preDisparate,preDisparate inAD,preDisparate inAD posCalibrated,...,preLFR posEqqOds,preLFR posROC,preReweighing,preReweighing inAD,preReweighing inAD posCalibrated,preReweighing inAD posEqqOds,preReweighing inAD posROC,preReweighing posCalibrated,preReweighing posEqqOds,preReweighing posROC
model,,,,,,,,,,,,,,,,,,,,,
adversarial,-0.4204,-0.1252,-0.3372,0.672,NaN,NaN,NaN,NaN,-0.2617,-0.2644,...,NaN,NaN,NaN,-0.1936,-0.4792,-0.4392,-0.2359,NaN,NaN,NaN
cat,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,-0.3828,NaN,NaN,...,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000
lgbm,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,-0.0063,NaN,NaN,...,NaN,NaN,0.0800,NaN,NaN,NaN,NaN,0.0800,0.0800,0.0800
lr,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,-0.0298,NaN,NaN,...,NaN,NaN,-0.0549,NaN,NaN,NaN,NaN,-0.0549,-0.0549,-0.0549
meta,NaN,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,...,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
metafair,NaN,NaN,NaN,NaN,NaN,0.0000,NaN,NaN,NaN,NaN,...,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mlp,NaN,NaN,NaN,NaN,0.0000,3.6968,3.3484,0.8098,NaN,NaN,...,3.3548,3.5335,0.0000,NaN,NaN,NaN,NaN,0.0001,0.0087,0.0000
tabnet,NaN,NaN,NaN,NaN,0.0000,0.0000,0.0000,0.8138,NaN,NaN,...,-0.2159,NaN,0.0000,NaN,NaN,NaN,NaN,-0.6825,-0.6825,0.0000
tabtransformer,NaN,NaN,NaN,NaN,0.5199,0.2977,0.3062,-0.1107,NaN,NaN,...,1.6555,1.7618,-0.1168,NaN,NaN,NaN,NaN,0.2168,NaN,-0.2567
